# 03 · Quantize int8 + so sánh 1 tầng vs 2 tầng + đo tốc độ CPU

Kaggle **CPU** (4 nhân) - cố ý không dùng GPU để số đo gần với laptop.
Input: code + VNTS + output notebook 01 (models/detector_*.onnx, datasets) + 02 (models/signnet*.onnx).
Secret `HF_TOKEN` để đẩy model lên HF Hub.

In [ ]:
import glob, json, os, shutil, sys

CODE = glob.glob("/kaggle/input/**/dashcam/__init__.py", recursive=True)[0].rsplit("/dashcam", 1)[0]
VNTS = [d for d in glob.glob("/kaggle/input/*") if "traffic" in d.lower()][0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
os.makedirs("models", exist_ok=True)
for f in glob.glob("/kaggle/input/**/models/*", recursive=True):
    shutil.copy(f, "models/")
!pip install -q onnx onnxruntime opencv-python-headless
!python -m dashcam.prepare_vnts --src {VNTS} --out datasets/vnts
!ls -la models

In [ ]:
!python -m dashcam.export --yolo models/detector_1cls.onnx --yolo models/detector_ncls.onnx \
    --calib-images datasets/vnts/yolo_ncls/images/train --imgsz 640 --out models

## mAP@0.5 trên test (ảnh tĩnh) + ms/ảnh với 4 luồng CPU

In [ ]:
!python -m dashcam.evaluate --images datasets/vnts/yolo_ncls/images/test --labels datasets/vnts/yolo_ncls/labels/test \
    --models models --class-stats datasets/vnts/class_stats.json --threads 4 --limit 400 --out results
print(open("results/eval.md", encoding="utf-8").read())

## 1 luồng (giống thiết bị nhúng yếu)

In [ ]:
!python -m dashcam.evaluate --images datasets/vnts/yolo_ncls/images/test --labels datasets/vnts/yolo_ncls/labels/test \
    --models models --threads 1 --limit 150 --out results_1thread
print(open("results_1thread/eval.md", encoding="utf-8").read())

In [ ]:
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
REPO = "DuongCodeAI/vn-dashcam-vision"
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="models", repo_id=REPO, allow_patterns=["*.onnx", "*.json"])
api.upload_folder(folder_path="results", repo_id=REPO, path_in_repo="results")
print(f"https://huggingface.co/{REPO}")